# 02 - Data Cleaning

This notebook performs data cleaning operations on the validated dataset. The goal is to improve data quality while preserving the integrity of the original dataset.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

## Load Validated Dataset

Load the raw dataset that successfully passed the validation phase. All cleaning operations will begin from this dataset.

In [2]:
DATA_PATH = Path("../data/raw/creditcard.csv")

df = pd.read_csv(DATA_PATH)

print(f"Original Shape: {df.shape}")

Original Shape: (284807, 31)


## Remove Duplicate Records

Identify and remove duplicate transactions while keeping the first occurrence. This helps eliminate redundant observations that could bias model training.

In [3]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate Records Found: {duplicate_count}")

Duplicate Records Found: 1081


In [4]:
df_clean = df.drop_duplicates().reset_index(drop=True)

print(f"Cleaned Shape: {df_clean.shape}")
print(f"Duplicates Removed: {duplicate_count}")

Cleaned Shape: (283726, 31)
Duplicates Removed: 1081


## Verify Duplicate Removal

Confirm that no duplicate records remain after the cleaning process.

In [5]:
remaining_duplicates = df_clean.duplicated().sum()

print(f"Remaining Duplicates: {remaining_duplicates}")

assert remaining_duplicates == 0, "Duplicate removal failed."

print("Duplicate removal completed successfully.")

Remaining Duplicates: 0
Duplicate removal completed successfully.


## Handle Missing Values

Inspect the cleaned dataset for missing values. If any missing values are detected, they should be handled appropriately. For this dataset, no missing values are expected.

In [6]:
missing_summary = df_clean.isnull().sum()

missing_summary = missing_summary[missing_summary > 0]

missing_summary

Series([], dtype: int64)

## Missing Value Summary

Summarize the total number and percentage of missing values after duplicate removal.

In [7]:
total_missing = df_clean.isnull().sum().sum()

missing_percentage = (
    total_missing / (df_clean.shape[0] * df_clean.shape[1])
) * 100

print(f"Total Missing Values : {total_missing}")
print(f"Missing Percentage   : {missing_percentage:.4f}%")

Total Missing Values : 0
Missing Percentage   : 0.0000%


In [8]:
assert total_missing == 0, "Missing values detected."

print("No missing values found. No imputation required.")

No missing values found. No imputation required.


## Handle Invalid Values

Inspect the dataset for invalid values such as infinite numbers, negative transaction amounts, and unexpected target labels. These checks help ensure data integrity before feature engineering.

## Infinite Value Check

Verify that the dataset does not contain positive or negative infinite values, which can cause failures during preprocessing and model training.

In [9]:
infinite_values = np.isinf(df_clean.select_dtypes(include="number")).sum().sum()

print(f"Infinite Values: {infinite_values}")

Infinite Values: 0


## Transaction Amount Validation

Ensure that all transaction amounts are valid. Since transaction amounts cannot be negative, verify that every value is zero or greater.

In [10]:
negative_amounts = (df_clean["Amount"] < 0).sum()

print(f"Negative Transaction Amounts: {negative_amounts}")

Negative Transaction Amounts: 0


## Target Label Validation

Confirm that the target variable contains only the expected binary classes representing legitimate and fraudulent transactions.

In [11]:
invalid_classes = df_clean.loc[
    ~df_clean["Class"].isin([0, 1]),
    "Class"
]

print(f"Invalid Target Labels: {len(invalid_classes)}")


Invalid Target Labels: 0


In [12]:
assert infinite_values == 0, "Infinite values detected."
assert negative_amounts == 0, "Negative transaction amounts detected."
assert len(invalid_classes) == 0, "Invalid target labels detected."

print("Invalid value validation passed.")

Invalid value validation passed.


## Outlier Analysis

Analyze the distribution of numerical features to identify potential outliers. In fraud detection, outliers often represent meaningful fraudulent behavior rather than noisy observations, so they are examined instead of being removed.

In [13]:
numeric_columns = df_clean.select_dtypes(include="number").columns

print(f"Numeric Features: {len(numeric_columns)}")
print(numeric_columns.tolist())

Numeric Features: 31
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


## Transaction Amount Distribution

Review descriptive statistics for the transaction amount feature to understand its spread and identify unusually large transactions.

In [14]:
df_clean["Amount"].describe()

count    283726.000000
mean         88.472687
std         250.399437
min           0.000000
25%           5.600000
50%          22.000000
75%          77.510000
max       25691.160000
Name: Amount, dtype: float64

## IQR-Based Outlier Detection

Estimate the number of potential outliers in the transaction amount feature using the Interquartile Range (IQR) method. This analysis is performed for reporting purposes only.

In [15]:
Q1 = df_clean["Amount"].quantile(0.25)
Q3 = df_clean["Amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

outliers = df_clean[
    (df_clean["Amount"] < lower_bound) |
    (df_clean["Amount"] > upper_bound)
]

print(f"Lower Bound : {lower_bound:.2f}")
print(f"Upper Bound : {upper_bound:.2f}")
print(f"Potential Outliers : {len(outliers)}")

Lower Bound : -102.27
Upper Bound : 185.38
Potential Outliers : 31685


## Cleaning Decision

Since transaction outliers may represent genuine fraudulent activity, they are retained in the dataset. Removing them could reduce the model's ability to detect fraud.

In [16]:
print("Decision: No outliers were removed from the dataset.")

Decision: No outliers were removed from the dataset.


## Data Consistency Checks

Perform final consistency checks on the cleaned dataset to ensure there are no unexpected issues before moving to feature engineering.

In [17]:
print(f"Dataset Shape : {df_clean.shape}")
print(f"Total Columns : {len(df_clean.columns)}")

display(df_clean.head())

Dataset Shape : (283726, 31)
Total Columns : 31


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Unique Value Summary

Review the number of unique values in each feature to identify unexpected constant or inconsistent columns.

In [18]:
unique_summary = pd.DataFrame({
    "Unique Values": df_clean.nunique()
})

unique_summary

,Unique Values
Time,124592
V1,275663
V2,275663
V3,275663
V4,275663
V5,275663
V6,275663
V7,275663
V8,275663
V9,275663


## Constant Feature Check

Identify features that contain only a single unique value. Such features do not contribute to model learning and may need to be removed during feature engineering.

In [19]:
constant_columns = unique_summary[
    unique_summary["Unique Values"] == 1
].index.tolist()

print(f"Constant Columns: {len(constant_columns)}")

if constant_columns:
    print(constant_columns)
else:
    print("No constant columns found.")

Constant Columns: 0
No constant columns found.


## Final Data Quality Status

Summarize the outcome of the complete data cleaning phase before proceeding to exploratory data analysis.

In [20]:
print("Data Cleaning Summary")
print("-" * 30)
print(f"Rows                : {df_clean.shape[0]}")
print(f"Columns             : {df_clean.shape[1]}")
print(f"Duplicate Records   : {df_clean.duplicated().sum()}")
print(f"Missing Values      : {df_clean.isnull().sum().sum()}")
print(f"Constant Columns    : {len(constant_columns)}")

print("\nDataset is clean and ready for Exploratory Data Analysis.")

Data Cleaning Summary
------------------------------
Rows                : 283726
Columns             : 31
Duplicate Records   : 0
Missing Values      : 0
Constant Columns    : 0

Dataset is clean and ready for Exploratory Data Analysis.
